In [4]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

In [2]:
def test_connection():
    with driver.session() as session:
        session.run("MATCH (n) RETURN n LIMIT 1")

try:
    test_connection()
    print("连接成功")
except Exception as e:
    print("连接失败",e)
finally:
    driver.close()        

连接成功


In [10]:
with driver.session() as session:
    session.run("""
                create (p1:Person{name:'zhangsan',age:30,city:'beijing'}),
                (p2:Person {name:'lisi',age:23,city:'shanghai'})
                """)
    
    session.run("""
                create (c1:Company {name:'shuzhiweilai',industry:'Technology',location:'beijing'}),
                (c2:Company {name:'naixue',industry:'Education',location:'beijing'})
                """)

In [11]:
with driver.session() as session:
    p1=session.run("""
                MATCH (p:Person{name:'zhangsan'})
                set p:Student
                return p
                """)
    p2=session.run("""
                match (p:Person{name:'lisi'})
                set p:Student
                return p
                """)
    print(p1)
    print(p2)

In [12]:
with driver.session() as session:
    session.run("""
                match (p:Person {name:'zhangsan'})
                set p.age=31
                """)

In [13]:
with driver.session() as session:
    session.run("""
                match (p:Person{name:'zhangsan'})
                match (c:Company{name:'shuzhiweilai'})
                create (p)-[:EMPLOYED_BY]->(c)
                """)
    
    session.run("""
                match(p:Person{name:'lisi'})
                match(c:Company{name:'naixue'})
                create (p)-[:EMPLOYED_BY]->(c)
                """)

In [9]:
import re


def neo4j_query_examples(driver,query_type,params=None):
    """
    执行各种Neo4j查询示例
    参数：
    - driver: Neo4j驱动实例
    - query_type: 查询类型，可选值包括：
    'all_persons', 'all_companies', 'filter_by_city', 'all_relationships',
    'specific_relationship', 'node_relationships', 'path_query', 'aggregation',
    'group_by', 'colleagues', 'complex_query', 'param_query', 'subgraph', 'community'
    - params: 查询参数字典，根据查询类型不同而不同
    返回：
    - 查询结果列表
    """
    if params is None:
        params={}
    results=[]
    
    with driver.session() as session:
        if query_type=='all_persons':
            result=session.run("""
            match (p:Person)
            return p.name as name, p.age as age,p.city as city
                               """)
            print("所有person节点：")
            for record in result:
                print(f"姓名：{record['name']},年龄：{record['age']},城市：{record['city']}")
                results.append({
                    'name':record['name'],
                    'age':record['age'],
                    'city':record['city']
                })
        elif query_type=='all_companies':
            result=session.run("""
                               match (c:Company)
                               return c.name as name,c.industry as industry,c.location as location
                               """)
            print("所有company节点")
            for record in result:
                print(f"公司：{record['name']},行业：{record['industry']},位置：{record['location']}")
                results.append({
                    'name':record['name'],
                    'industry':record['industry'],
                    'location':record['location']
                })
        elif query_type=='filter_by_city':
            city=params.get('city','beijing')
            result=session.run("""
                               match (p:Person)
                               where p.city=$city
                               return p.name as name,p.age as age
                               """,{'city':city})
            print(f"{city}人员")
            for record in result:
                print(f"姓名：{record['name']},年龄：{record['age']}")
                results.append({
                    'name':record['name'],
                    'age':record['age']
                })
        elif query_type=='all_relationships':
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               return p.name as person, type(r) as relationship, c.name as company
                               """)
            print("所有人员与公司的关系：")
            for record in result:
                print(f"{record['person']} {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })
        elif query_type=='specific_relationship':
            rel_type=params.get('rel_type','EMPLOYED_BY')
            result=session.run("""
                               match (p:Person)-[:{rel_type}]->(c:Company)
                               return p.name as person, c.name as company
                               """)
            print(f"{rel_type}关系")
            for record in result:
                print(f"{record['person']} 与 {record['company']} 有 {rel_type}关系")
                results.append({
                    'person':record['person'],
                    'company':record['company']
                })
        elif query_type=='node_relationships':
            person_name=params.get('person_name','zhangsan')
            result=session.run("""
                               match (p:Person{name:$name})-[r]->(c)
                               return type(r) as relationship,c.name as connected_to,labels(c) as node_type
                               """,{'name':person_name})
            print(f"{person_name}的所有关系")
            for record in result:
                print(f"关系类型：{record['relationship']},连接到: {record['connected_to']},节点类型：{record['node_type']}")
                results.append({
                    'relationship':record['relationship'],
                    'connected_to':record['connected_to'],
                    'node_type':record['node_type']
                })
        elif query_type=='aggregation':
            result=session.run("""
                               match (p:Person)-[:EMPLOYED_BY]->(c:Company)
                               return c.name as company,count(p) as employee_count,avg(p.age) as avg_age
                               """)
            print("公司员工统计：")
            for record in result:
                print(f"公司：{record['company']},员工数：{record['employee_count']},平均年龄：{round(record['avg_age'],1)}")
                results.append({
                    'company':record['company'],
                    'employee_count':record['employee_count'],
                    'avg_age':record['avg_age'],
                })
        elif query_type=='complex_query':
            min_age=params.get('min_age',25)
            location=params.get('location','beijing')
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               where p.age >$min_age and c.location=$location
                               and (type(r)='EMPLOYED_BY' or type(r)='INVESTED_IN')
                               return p.name as person, p.age as age,
                                type(r) as relationship,c.name as company
                                order by p.age desc
                               """,{'min_age':min_age,'location':location})
            
            print(f"{min_age}岁以上且与{location}公司有雇佣或投资关系的人：")
            for record in result:
                print(f"{record['person']} ({record['age']}岁) {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'age':record['age'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })
        elif query_type=='param_query':
            query_params={
                'min_age':params.get('min_age',25),
                'location':params.get('location','beijing'),
                'relationship_types':params.get('relationship_types',['EMPLOYED_BY','INVESTED_BY'])
            }
            
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               where p.age >$min_age and c.location= $location
                               and type(r) in $relationship_types
                               return p.name as person, type(r) as relationship, c.name as company
                               
                               """,query_params)
            
            print(f"参数化查询结果（年龄》{query_params['min_age']}, 位置{query_params['location']}:")
            for record in result:
                print(f"{record['person']} {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })      
        else:
            print(f"未知的查询类型：{query_type}")
            results.append({'error':f"未知的查询类型：{query_type}"})
    return results

In [10]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

try:
    neo4j_query_examples(driver,'all_persons')
    
    neo4j_query_examples(driver,'filter_by_city',{'city':'beijing'})
    neo4j_query_examples(driver,'node_relationships',{'person_name':'zhangsan'})
    neo4j_query_examples(driver,'complex_query',{'min_age':20,'location':'beijing'})
finally:
    driver.close()

所有person节点：
姓名：zhangsan,年龄：31,城市：beijing
姓名：lisi,年龄：23,城市：shanghai
beijing人员
姓名：zhangsan,年龄：31
zhangsan的所有关系
关系类型：EMPLOYED_BY,连接到: shuzhiweilai,节点类型：['Company']
20岁以上且与beijing公司有雇佣或投资关系的人：
zhangsan (31岁) EMPLOYED_BY shuzhiweilai
lisi (23岁) EMPLOYED_BY naixue
